In [1]:
! pip install langchain langchain-core langchain-cohere langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 225.9 kB/s  0:00:14
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 186.3 kB/s  0:00:12
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 253.1 kB/s  0:00:03
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [langchain-community]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
! pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 178.7 kB/s  0:00:36
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ddgs]


In [3]:
! pip install sentence-transformers

In [4]:
! pip install pypdf langchain-text-splitters faiss-cpu

In [6]:
from google.colab import userdata
COHERE_API_KEY = userdata.get('COHERE_API_KEY')

In [7]:
import os
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=COHERE_API_KEY
)

response = llm.invoke(
    "What is GenAI answer me in 3 words"
)
print(response.content)


Generative Artificial Intelligence.


In [ ]:
# ! pip install -r requirements.txt

In [ ]:
# ----------------------------------
#  python -m venv product_version1
#  product_version1/scripts/activate
#
#  pip install pandas == 3.2
#  pip install langchain == 1.2.4
#
# file: requirements.txt
# pandas == 3.2.0
# pyspark == 3.5.1
# langchain >= 1.0
#
# ..
# (product_version1)  pip install -r requirements.txt
#
# deactivate
# C:\> pip show pandas
# No module Found
# C:\> pip install pandas
# ..
# C:\> pip show pandas    <== Global Env
# pandas ... 4.5.2 version

In [8]:
# Cohere
from langchain_cohere import CohereEmbeddings

embeddings = CohereEmbeddings(model="embed-v5.0-pro",cohere_api_key=COHERE_API_KEY)

text = "Hello"
query_result = embeddings.embed_query(text)
print(len(query_result))

2048


In [14]:
! pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 74.7 MB/s  0:00:00


In [9]:
def fx(app_name):
  return f"App name is:{app_name}"


fx("webApp") # this is normal function

'App name is:webApp'

In [10]:
from langchain_core.runnables import RunnableLambda
def fx(app_name):
  return f"App name is:{app_name}"


obj = RunnableLambda(fx)

result = obj.invoke('webApp')
print(result)

App name is:webApp


In [11]:
def f1(text):
  return text.lower()

def f2(text):
  return "Customer question:" + text


obj1 = RunnableLambda(f1)
obj2 = RunnableLambda(f2)

chain = obj1|obj2
result = chain.invoke("WHAT IS A LOAN")
print(result)

Customer question:what is a loan


In [16]:
# Extend RAG + ChatHistory ->DB
# |
# Rerank
# -------
# |
# oracle23AI DB
# |
# UI
#--------

# Step 1: Load PDF
from langchain_community.document_loaders import PyPDFLoader
loaderObj = PyPDFLoader('attention.pdf')
docs = loaderObj.load()

# Step 2: Split PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitterObj = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 50)
chunks = splitterObj.split_documents(docs)


# Step 3: Create Embeddings
from langchain_cohere import CohereEmbeddings
embedObj = CohereEmbeddings(model="embed-v5.0-pro",cohere_api_key=COHERE_API_KEY)

# Step 4: Create FAISS Vector Store
#
# ! pip install langchain-chroma chromadb
# ------------------------------------------
from langchain_community.vectorstores import FAISS
vector_stores = FAISS.from_documents(chunks,embedObj)
retriever_obj = vector_stores.as_retriever()


# Step 5: Create LLM
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=COHERE_API_KEY
)

In [22]:

# Step 6: Create Strict Prompt - ChatPromptTemplate,MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
my_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Act as strict QA assistnt
        Answer only using the PDF context.
        if the answer is not present in the context, reply exactly:
        I don't Know, the document does not contains this information
        PDF Context:
        {context}"""
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}")
])

# Step 7: Create a simple RAG Function
#          - using retiever object search PDF chontents
#          - invoke llm  get the response
#          - return response content
def answer_question(inputs):
  question = inputs['question']
  history = inputs['history']

  # Search the PDF
  documents = retriever_obj.invoke(question)

  context = ""
  for doc in documents:
    context = context + doc.page_content + "\n"

  # prepare the prompt
  prompt_value = my_prompt.invoke({"question":question,"context":context,"history":history})

  # ask the LLM
  response = llm.invoke(prompt_value)
  return response.content


# Step 8: connect RAG with ChatHistory
#          - Refer DAY2 code
#            stores = {}
#            ...
from langchain_core.runnables import RunnableLambda,RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

rag_chain = RunnableLambda(answer_question)

stores = {}
def get_session_history(session_id: str) ->BaseChatMessageHistory:
  if session_id not in stores:
    stores[session_id] = ChatMessageHistory()
  return stores[session_id]

chatbot = RunnableWithMessageHistory(rag_chain,get_session_history,
                                     input_messages_key="question",
                                     history_messages_key="history")

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [23]:
# Step 9: Chat with the PDF

session_id = "chat01"
while True:
  question = input("\n Enter your question: ")
  if question.lower() == "exit":
    break

  response = chatbot.invoke(
      {"question": question},
      config = {"configurable":{"session_id":session_id}
      }
  )
  print("\n Assistant:",response )




 Enter your question: How to write Hello world Ruby program

 Assistant: I don't Know, the document does not contains this information.

 Enter your question: tell me about self-attention

 Assistant: *6

 Enter your question: explain again

 Assistant: The document describes self-attention in the context of multi-head attention within a neural network model. Here's a breakdown based on the provided text:

**Multi-Head Attention:**

* **Purpose:** Multi-head attention allows the model to focus on different aspects of the input data simultaneously. It achieves this by using multiple "attention heads," each learning to attend to different parts of the input.
 * **Process:**  The input (queries, keys, and values) are transformed through linear projections for each attention head. This creates multiple representations of the input, each capturing different information.
 * **Attention Calculation:** Each head calculates attention scores between the projected queries and keys, determining t

In [24]:
# Step 9: Chat with the PDF

session_id = "chat02"
while True:
  question = input("\n Enter your question: ")
  if question.lower() == "exit":
    break

  response = chatbot.invoke(
      {"question": question},
      config = {"configurable":{"session_id":session_id}
      }
  )
  print("\n Assistant:",response )



 Enter your question: what is my previous question

 Assistant: I don't Know, the document does not contains this information.

 Enter your question: exit


In [26]:
stores

{'chat01': InMemoryChatMessageHistory(messages=[HumanMessage(content='How to write Hello world Ruby program', additional_kwargs={}, response_metadata={}), AIMessage(content="I don't Know, the document does not contains this information.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='tell me about self-attention', additional_kwargs={}, response_metadata={}), AIMessage(content='*6', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='explain again', additional_kwargs={}, response_metadata={}), AIMessage(content='The document describes self-attention in the context of multi-head attention within a neural network model. Here\'s a breakdown based on the provided text:\n\n**Multi-Head Attention:**\n\n* **Purpose:** Multi-head attention allows the model to focus on different aspects of the input data simultaneously. It achieves this by using multiple "attention heads," each learnin

In [ ]:
#------------------------------------------------
# [Python] -------------------[Database]
#   - native class                - SQL
#
# "select *from table"            sql> select *from table
# |___________________|                ---------------------//Query
#   // string - str

# step 1: import <DB-Module>   (ex:  import oracledb  ;  import sqlite3 ; import connection.mysqldb)
# |
# step 2: <DB-Module>.connect() -> connectionObject
# |
# step 3: connectionObject.cursor() ->cursorObject
# |
# step 4: cursorObject.execute(DBQuery)
#                      -----------------
#                               |->select query ->fetchone() / fetchall() ->[(),(),(),..()] / fetchmany(n=3)
# step 5: connectionObject.commit()
# step 6: connectionObject.close()
# ------------------------------------------------------------------------------------------------------------

In [27]:
import sqlite3
sqlite3.connect("products.db")

In [28]:
conn = sqlite3.connect("products.db")
conn.cursor()

In [29]:
sth = conn.cursor()
sth.execute("create table products(id INT,pname TEXT,pcost INT)")

In [32]:
sth.execute("insert into products(id,pname,pcost) values(101,'pA',1000)")

In [33]:
sth.execute("insert into products(id,pname,pcost) values(102,'pB',2000)")
sth.execute("insert into products(id,pname,pcost) values(103,'pC',3000)")
sth.execute("insert into products(id,pname,pcost) values(104,'pD',1400)")
sth.execute("insert into products(id,pname,pcost) values(105,'pE',4000)")

In [34]:
va = 106
vb = 'pF'
vc = 3500.32
sth.execute("insert into products(id,pname,pcost) values(?,?,?)",(va,vb,vc))

In [35]:
sth.execute("select *from products")

In [36]:
sth.fetchone()

(101, 'pA', 1000)

In [37]:
sth.fetchone()

(102, 'pB', 2000)

In [38]:
sth.execute("select *from products")
sth.fetchall()

[(101, 'pA', 1000),
 (102, 'pB', 2000),
 (103, 'pC', 3000),
 (104, 'pD', 1400),
 (105, 'pE', 4000),
 (106, 'pF', 3500.32)]

In [39]:
sth.execute("select *from products")
list(sth) # typecast to list - generator

[(101, 'pA', 1000),
 (102, 'pB', 2000),
 (103, 'pC', 3000),
 (104, 'pD', 1400),
 (105, 'pE', 4000),
 (106, 'pF', 3500.32)]

In [40]:
conn.commit()

In [41]:
conn.close()

In [42]:
conn = sqlite3.connect("products.db")
sth = conn.cursor()
sth.execute("select *from products where pcost > 2000")
list(sth)

[(103, 'pC', 3000), (105, 'pE', 4000), (106, 'pF', 3500.32)]

In [44]:
## demo - chat1.db

conn = sqlite3.connect("chat1.db")
sth = conn.cursor()
# create table
sth.execute("""
create table chat_history(
id INTEGER PRIMARY KEY AUTOINCREMENT,
session_id TEXT,
message TEXT)""")

va = "chat01"
vb = "what is self-attention"
# insert a message
sth.execute("insert into chat_history(session_id,message) values(?,?)",(va,vb))

In [45]:
conn.commit() # save changes

In [46]:
conn.close()

In [47]:
conn = sqlite3.connect("chat1.db")
sth = conn.cursor()
sth.execute("select *from chat_history")
list(sth)

[(1, 'chat01', 'what is self-attention')]

In [ ]:
## SQLAlchemy - Python tool kit
#  -----------
#  sqlalachemy[asyncio]
#  -----------------------


# User
# AI
# -------------// DB
#
#
from langchain_community.chat_message_histories import SQLChatMessageHistory
history = SQLChatMessageHistory(session_id = "chat01",connection_string="sqlite:///chat1.db")

In [48]:
import sqlite3

# connect to database
conn = sqlite3.connect("chat_history.db")

# create a table
sth = conn.cursor()
sth.execute("""
create table if not exists chat_history(
id INTEGER PRIMARY KEY AUTOINCREMENT,
session_id TEXT,
role TEXT,
message TEXT)""")

# insert a message
sth.execute("insert into chat_history(session_id,role,message) values(?,?,?)",("chat01","user","what is nltk?"))

# save changes and close
conn.commit()
conn.close()

In [50]:
! pip install sqlalchemy[asyncio]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 16.0 MB/s  0:00:00


In [ ]:
# Step 1 : Load PDF


# Step 2: Split PDF into Chunks

# Step 3: Create Embeddings


# Step 4: Create Vector Stores

# Step 5: Create LLM

# Step 6: Create strict prompt

# Step 7: Create RAG Function
          # get the user question and history
          # search the pdf - get the docs
          # prepare the prompt
          # invoke to llm
          # return response content

# Step 8 : Connect RAG with SQLite Chat History
from langchain_core.runnables import RunnableLambda,RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

# Return chat history from sqliteDB
def get_session_history(session_id: str):
  history = SQLChatMessageHistory(session_id=session_id,
                        connection_string="sqlite:///chat_history.db")
  return history


rag_chain = RunnableLambda(answer_question)
chatbot = RunnableWithMessageHistory(rag_chain,get_session_history,
                           input_messages_key="question",
                           history_messages_key="history")

In [51]:
# Step 1: Load PDF
from langchain_community.document_loaders import PyPDFLoader
loaderObj = PyPDFLoader('attention.pdf')
docs = loaderObj.load()

# Step 2: Split PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitterObj = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 50)
chunks = splitterObj.split_documents(docs)


# Step 3: Create Embeddings
from langchain_cohere import CohereEmbeddings
embedObj = CohereEmbeddings(model="embed-v5.0-pro",cohere_api_key=COHERE_API_KEY)

# Step 4: Create FAISS Vector Store
#
# ! pip install langchain-chroma chromadb
# ------------------------------------------
from langchain_community.vectorstores import FAISS
vector_stores = FAISS.from_documents(chunks,embedObj)
retriever_obj = vector_stores.as_retriever()


# Step 5: Create LLM
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=COHERE_API_KEY
)

In [52]:
# Step 6: Create Strict Prompt - ChatPromptTemplate,MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
my_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Act as strict QA assistnt
        Answer only using the PDF context.
        if the answer is not present in the context, reply exactly:
        I don't Know, the document does not contains this information
        PDF Context:
        {context}"""
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}")
])

# Step 7: Create a simple RAG Function
#          - using retiever object search PDF chontents
#          - invoke llm  get the response
#          - return response content
def answer_question(inputs):
  question = inputs['question']
  history = inputs['history']

  # Search the PDF
  documents = retriever_obj.invoke(question)

  context = ""
  for doc in documents:
    context = context + doc.page_content + "\n"

  # prepare the prompt
  prompt_value = my_prompt.invoke({"question":question,"context":context,"history":history})

  # ask the LLM
  response = llm.invoke(prompt_value)
  return response.content

In [56]:
# Step 8 : Connect RAG with SQLite Chat History
from langchain_core.runnables import RunnableLambda,RunnableWithMessageHistory
from langchain_community.chat_message_histories import SQLChatMessageHistory

# Return chat history from sqliteDB
def get_session_history(session_id: str):
  history = SQLChatMessageHistory(session_id=session_id,
                        connection="sqlite:///chat_history.db")
  return history


rag_chain = RunnableLambda(answer_question)
chatbot = RunnableWithMessageHistory(rag_chain,get_session_history,
                           input_messages_key="question",
                           history_messages_key="history")

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [57]:
# Step 9: Chat with the PDF
# ----------------------------
session_id = "chat02"

while True:
  question = input("Enter your question:")
  if question.lower() == "exit":
    break

  if question.lower() == "history":
    history = get_session_history(session_id)
    print(history)
    continue
  response = chatbot.invoke({"question":question},config={"configurable":{"session_id":session_id}})
  print("\nAssistant:",response)
  print("\n")


Enter your question:what is self-attention

Assistant: Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. In the context of the provided document, self-attention is used in both the encoder and decoder layers of the Transformer model. 

- **In the encoder**, each position can attend to all positions in the previous layer of the encoder, with keys, values, and queries all coming from the output of the previous layer.  
- **In the decoder**, each position can attend to all positions in the decoder up to and including that position, with leftward attention prevented.  

The self-attention mechanism is computed using the formula:  
**Attention(Q, K, V) = softmax(QK^T / √dk)V**, where Q, K, and V represent queries, keys, and values, respectively, and dk is the dimension of the keys.


Enter your question:tell me in simple words

Assistant: Self-attention is a way 

In [55]:
#help(SQLChatMessageHistory)

Help on class SQLChatMessageHistory in module langchain_community.chat_message_histories.sql:

class SQLChatMessageHistory(langchain_core.chat_history.BaseChatMessageHistory)
 |  SQLChatMessageHistory(
 |      session_id: str,
 |      table_name: str = 'message_store',
 |      session_id_field_name: str = 'session_id',
 |      custom_message_converter: Optional[langchain_community.chat_message_histories.sql.BaseMessageConverter] = None,
 |      connection: Union[NoneType, sqlalchemy.ext.asyncio.engine.AsyncEngine, sqlalchemy.engine.base.Engine, str] = None,
 |      engine_args: Optional[Dict[str, Any]] = None,
 |      async_mode: Optional[bool] = None
 |  )
 |
 |  Chat message history stored in an SQL database.
 |
 |  Example:
 |      .. code-block:: python
 |
 |          from langchain_core.messages import HumanMessage
 |
 |          from langchain_community.chat_message_histories import SQLChatMessageHistory
 |
 |          # create sync sql message history by connection_string
 |    

In [58]:
# Step 9: Chat with the PDF
# ----------------------------
session_id = "chat03"

while True:
  question = input("Enter your question:")
  if question.lower() == "exit":
    break

  if question.lower() == "history":
    history = get_session_history(session_id)
    print(history)
    continue
  response = chatbot.invoke({"question":question},config={"configurable":{"session_id":session_id}})
  print("\nAssistant:",response)
  print("\n")

Enter your question:history

Enter your question:tell me my previous question

Assistant: I don't Know, the document does not contains this information.


Enter your question:explain what is transformer tell me in 10 words

Assistant: Transformer: Attention-based model, no recurrence, faster training, global dependencies.


Enter your question:history
Human: tell me my previous question
AI: I don't Know, the document does not contains this information.
Human: explain what is transformer tell me in 10 words
AI: Transformer: Attention-based model, no recurrence, faster training, global dependencies.
Enter your question:exit


In [63]:
# Step 9: Chat with the PDF
# ----------------------------
session_id = "chat05"

while True:
  question = input("Enter your question:")
  if question.lower() == "exit":
    break

  if question.lower() == "history":
    history = get_session_history(session_id)
    print(history)
    continue
  response = chatbot.invoke({"question":question},config={"configurable":{"session_id":session_id}})
  print("\nAssistant:",response)
  print("\n")

Enter your question:history

Enter your question:what is the latest version of langchain

Assistant: I don't Know, the document does not contains this information.


Enter your question:exit


In [64]:
## For Testing

import sqlite3
myconn = sqlite3.connect("chat_history.db")
sth = myconn.cursor()
sth.execute("select *from chat_history")
for var in sth:
  print(var)

(1, 'chat01', 'user', 'what is nltk?')


In [65]:

import sqlite3

myconn = sqlite3.connect("chat_history.db")
sth = myconn.cursor()

# Display all tables in the database
sth.execute("""
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
""")

print("Tables in database:")
for row in sth.fetchall():
    print(row)

myconn.close()

Tables in database:
('chat_history',)
('sqlite_sequence',)
('message_store',)


In [66]:
import sqlite3

conn = sqlite3.connect("chat_history.db")
cursor = conn.cursor()

cursor.execute("""
    SELECT session_id, message
    FROM message_store
    WHERE session_id = ?
""", ("chat02",))

for session_id, message in cursor.fetchall():
    print("Session:", session_id)
    print("Message:", message)
    print("-" * 50)

conn.close()


Session: chat02
Message: {"type": "human", "data": {"content": "what is self-attention", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null}}
--------------------------------------------------
Session: chat02
Message: {"type": "ai", "data": {"content": "Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. In the context of the provided document, self-attention is used in both the encoder and decoder layers of the Transformer model. \n\n- **In the encoder**, each position can attend to all positions in the previous layer of the encoder, with keys, values, and queries all coming from the output of the previous layer.  \n- **In the decoder**, each position can attend to all positions in the decoder up to and including that position, with leftward attention prevented.  \n\nThe self-attention mechanism is computed using the f

In [67]:
import sqlite3

conn = sqlite3.connect("chat_history.db")
cursor = conn.cursor()

cursor.execute("PRAGMA table_info(message_store)")
print("Columns:", cursor.fetchall())

cursor.execute("SELECT * FROM message_store")
rows = cursor.fetchall()

print("\nTotal records:", len(rows))

for row in rows:
    print(row)

conn.close()


Columns: [(0, 'id', 'INTEGER', 1, None, 1), (1, 'session_id', 'TEXT', 0, None, 0), (2, 'message', 'TEXT', 0, None, 0)]

Total records: 16
(1, 'chat02', '{"type": "human", "data": {"content": "what is self-attention", "additional_kwargs": {}, "response_metadata": {}, "type": "human", "name": null, "id": null}}')
(2, 'chat02', '{"type": "ai", "data": {"content": "Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. In the context of the provided document, self-attention is used in both the encoder and decoder layers of the Transformer model. \\n\\n- **In the encoder**, each position can attend to all positions in the previous layer of the encoder, with keys, values, and queries all coming from the output of the previous layer.  \\n- **In the decoder**, each position can attend to all positions in the decoder up to and including that position, with leftward attent

In [ ]:
# JSON
# -----
# Storing and exchanging data - between webApplications
# ---------------------------

# json - module
#
# json.dumps(python_data) ->json  # convert from python ->json
# json.loads(json_data) ->python  # convert from json ->python
# ----------------

# python                           json
# ---------                       --------
# int ,float                       int,float
# str                               str
# True False                         true false
# list []                            array []
# tuple ()                           array []
# dict {'key':value}                 object {'key':value}


In [72]:
import json
json.dumps(True) # Converting from python to json

'true'

In [73]:
json_data = json.dumps(True)
json.loads(json_data) # Conver from json to Python

True

In [71]:
import sqlite3
import json

conn = sqlite3.connect("chat_history.db")
cursor = conn.cursor()

cursor.execute("""
select session_id,message from message_store order by id""")

print("=" * 70)
print("   CHATBOT CONVERSATION HISTORY  ")
print("=" * 70)

for session_id,message_json in cursor.fetchall():
  msg = json.loads(message_json) # convert from json to python
  role = msg['type'].upper()
  content = msg['data']['content']
  print(f"\n Session ID:{session_id}")
  print(f"{role}:{content}")
  print("-"*70)

conn.close()

   CHATBOT CONVERSATION HISTORY  

 Session ID:chat02
HUMAN:what is self-attention
----------------------------------------------------------------------

 Session ID:chat02
AI:Self-attention, sometimes called intra-attention, is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. In the context of the provided document, self-attention is used in both the encoder and decoder layers of the Transformer model. 

- **In the encoder**, each position can attend to all positions in the previous layer of the encoder, with keys, values, and queries all coming from the output of the previous layer.  
- **In the decoder**, each position can attend to all positions in the decoder up to and including that position, with leftward attention prevented.  

The self-attention mechanism is computed using the formula:  
**Attention(Q, K, V) = softmax(QK^T / √dk)V**, where Q, K, and V represent queries, keys, and values, respectivel

In [ ]:
####
# Reranking in RAG
# ----------------
#
# In RAG
# UserQuestion -------> Retriever - retrieve top 10-20 chunks from vectordatabase
#                                       |
#                    *   Rerank  - each query-documents pair and sorts the chunks by relevance
#                                       |
#                                     5-7 chunks
#                                   ----------
#                                      |
#                                 pass only the best 5-7 chunks to the LLM to generate answer
#

In [75]:
# Step 1: Load PDF
from langchain_community.document_loaders import PyPDFLoader
loaderObj = PyPDFLoader('attention.pdf')
docs = loaderObj.load()
docs[0]

Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@google.com\nLlion Jones∗\nGoogle Research\nllion@google.com\nAidan N. Gomez∗ †\nUniversity of Toronto\naidan@cs.toronto.edu\nŁukasz Kaiser∗\nGoogle Brain\nlukaszka

In [78]:
from langchain_core.documents import Document

query = "what is the fee for international card transactions?"

docs = [ Document(page_content=(
    "Domestic ATM withdrawls have a fixed"
    "service charge according to the card type."
),metadata = {"source":"domestic_policy.pdf"}),
Document(page_content=(
    "International card transcation incur a"
    "2% transaction fee on the transction amount."
),metadata = {"source":"international_policy.pdf"}),
Document(page_content=(
    "currency conversation rates are determined"
    "fees are applicable."
),metadata = {"source":"currency_policy.pdf"}),
Document(page_content=(
    "certain eligible international transcation"
    "may qualify for fee exemptions under the policy"
),metadata = {"source":"exemption_policy.pdf"})
]

from langchain_cohere import CohereRerank

rerankObj  = CohereRerank(
    model="rerank-v5.0-pro",
    cohere_api_key=COHERE_API_KEY,
    top_n = 2
)

In [79]:
rerankObj.compress_documents(documents=docs,query=query)

[Document(metadata={'source': 'international_policy.pdf', 'relevance_score': 0.8070014}, page_content='International card transcation incur a2% transaction fee on the transction amount.'),
 Document(metadata={'source': 'exemption_policy.pdf', 'relevance_score': 0.32782567}, page_content='certain eligible international transcationmay qualify for fee exemptions under the policy')]

In [80]:
reranked_docs = rerankObj.compress_documents(documents=docs,query=query)

In [83]:
# Pass the reranked documents to an LLM
# Create LLM
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=COHERE_API_KEY
)

context = "\n\n".join(doc.page_content for doc in reranked_docs)

prompt = f"""
You are a bank policy assistant.and
Answer the user question using only
the provided policy context.
If the context is insufficient, say
Do not invent fees or policy exceptions.

Policy context:
{context}
Question:
{query}
Answer:
"""

response = llm.invoke(prompt)
print(response.content)

The fee for international card transactions is 2% of the transaction amount. Certain eligible international transactions may qualify for fee exemptions under the policy.


In [ ]:
## Sample Documents -->reranker ->Top 2 relevant chunks -> LLM ->FinalAnswer

In [ ]:
# PDF ->Chunks ->Embeddings ->VectorStores ===>Reranker ===>Prompt + ChatHistory --> LLM -->Answer
# ---------------------------------------- ==============   ===== ---------------------------------
                            #  |                |__ top_n = 3
                            #  |
                            # vector_stores.as_retriever(search_kwargs={"k":10})

In [ ]:
# retriever_docs = vector_stores.as_retriever(search_kwargs={"k":10})
# reranker = CohereRerank(model="<modelName>",cohere_api_key=COHERE_API_KEY,top_n=3)
# |
# inside our answer_question():
#
# reranked_docs = reranker.compress_documents(documents=retriever_docs,query="<userQuestion>")
# -------------
#

In [88]:
# Step 1: Load PDF
from langchain_community.document_loaders import PyPDFLoader
loaderObj = PyPDFLoader('attention.pdf')
docs = loaderObj.load()

# Step 2: Split PDF into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitterObj = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 50)
chunks = splitterObj.split_documents(docs)


# Step 3: Create Embeddings
from langchain_cohere import CohereEmbeddings
embedObj = CohereEmbeddings(model="embed-v5.0-pro",cohere_api_key=COHERE_API_KEY)

# Step 4: Create FAISS Vector Store
#
# ! pip install langchain-chroma chromadb
# ------------------------------------------
from langchain_community.vectorstores import FAISS
vector_stores = FAISS.from_documents(chunks,embedObj)
retriever_obj = vector_stores.as_retriever(search_kwargs={"k":10})


# Step 5: Create Reranker
from langchain_cohere import CohereRerank
reranker = CohereRerank(
    model="rerank-v5.0-pro",
    cohere_api_key=COHERE_API_KEY,
    top_n = 3
)

# Step 6: Create LLM
from langchain_cohere import ChatCohere

llm = ChatCohere(
    model="command-a-03-2025",
    cohere_api_key=COHERE_API_KEY
)

In [89]:


# Step 7: Create Strict Prompt
# Create Strict Prompt - ChatPromptTemplate,MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
my_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """Act as strict QA assistnt
        Answer only using the PDF context.
        if the answer is not present in the context, reply exactly:
        I don't Know, the document does not contains this information
        PDF Context:
        {context}"""
    ),
    MessagesPlaceholder(variable_name="history"),
    ("human","{question}")
])


# Step 8: RAG Function with Reranking
def answer_question(inputs):
  question = inputs['question']
  history = inputs['history']

  # 8a. Retrieve chunks
  retrieved_docs = retriever_obj.invoke(question)
  print(f"\nRetrieved chunks:{len(retrieved_docs)}")

  # 8b. Rerank
  reranked_docs = reranker.compress_documents(documents=retrieved_docs,query=question)
  print(f"\nChunks after rerank:{len(retrieved_docs)}")

  # 8c. Build context using only top-ranked chunks
  context = "\n\n".join(doc.page_content for doc in reranked_docs)

  # 8d. prepare the prompt
  prompt_value = my_prompt.invoke({"question":question,"context":context,"history":history})

  # 8e. ask the LLM
  response = llm.invoke(prompt_value)
  return response.content


In [90]:
# Step 9: Connect RAG with Chat History
#------------------------------------------
from langchain_core.runnables import RunnableLambda,RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

rag_chain = RunnableLambda(answer_question)

stores = {}
def get_session_history(session_id: str) ->BaseChatMessageHistory:
  if session_id not in stores:
    stores[session_id] = ChatMessageHistory()
  return stores[session_id]

chatbot = RunnableWithMessageHistory(rag_chain,get_session_history,
                                     input_messages_key="question",
                                     history_messages_key="history")

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [91]:
# Step 10: Chat with the PDF
# ----------------------------
session_id = "chat01"
while True:
  question = input("\n Enter your question (type 'exit' to stop): ")
  if question.lower() == "exit":
    break

  response = chatbot.invoke(
      {"question": question},
      config = {"configurable":{"session_id":session_id}
      }
  )
  print("\n Assistant:",response )


 Enter your question (type 'exit' to stop): what is self-attention

Retrieved chunks:10

Chunks after rerank:10

 Assistant: Self-attention, sometimes called intra-attention, is an attention mechanism that relates different positions of a single sequence to compute a representation of the sequence. It is used in both the encoder and decoder layers of a network. In the encoder, all keys, values, and queries come from the same place (the output of the previous layer), and each position can attend to all positions in the previous layer. In the decoder, self-attention allows each position to attend to all positions up to and including that position, with a need to prevent leftward information flow. Self-attention layers connect all positions with a constant number of sequentially executed operations, making them computationally faster than recurrent layers for longer sequences.

 Enter your question (type 'exit' to stop): explain transformer work flow

Retrieved chunks:10

Chunks after re

In [92]:
retriever_obj.invoke(question)

[Document(id='6cde0157-43c7-4073-8aa1-30646553e5f0', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 11, 'page_label': '12'}, page_content='layer. arXiv preprint arXiv:1701.06538, 2017.\n[33] Nitish Srivastava, Geoffrey E Hinton, Alex Krizhevsky, Ilya Sutskever, and Ruslan Salakhutdi-\nnov. Dropout: a simple way to prevent neural networks from overfitting. Journal of Machine\nLearning Research, 15(1):1929–1958, 2014.\n[34] Sainbayar Sukhbaatar, Arthur Szlam, Jason Weston, and Rob Fergus. End-to-end memory\nnetworks. In C. Cortes, N. D. Lawrence, D. D. Lee, M. Sugiyama, and R. Garnett, editors,'),
 Document(id='79333c6f-1510-4b16-8cec-17e4b

In [103]:
# Retrieve from FAISS
retrieved_docs = retriever_obj.invoke(question)

print("\n===== BEFORE RERANKING =====")
print("Number of chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs, start=1):
    print(f"\nChunk {i}")
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Content:", doc.page_content[:250])


print("user question:",question)
# Rerank with Cohere
reranked_docs = reranker.compress_documents(
    documents=retrieved_docs,
    query=question
)

print("\n===== AFTER RERANKING =====")
print("Number of chunks:", len(reranked_docs))

for i, doc in enumerate(reranked_docs, start=1):
    print(f"\nRank {i}")
    print("Relevance score:",
          doc.metadata.get("relevance_score"))
    print("Source:", doc.metadata.get("source"))
    print("Page:", doc.metadata.get("page"))
    print("Content:", doc.page_content[:250])


===== BEFORE RERANKING =====
Number of chunks: 10

Chunk 1
Source: attention.pdf
Page: 13
Content: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my

Chunk 2
Source: attention.pdf
Page: 14
Content: Input-Input Layer5
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my
opinion
.
<EOS>
<pad>
The
Law
will
never
be
perfect
,
but
its
application
should
be
just
-
this
is
what
we
are
missing
,
in
my

Chunk 3
Source: attention.pdf
Page: 3
Content: queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the
query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function

Chunk 4
Source: attention.pdf
Page: 0
Content: ‡W

In [ ]:
## Oracle23ai vector search
# ---------------------------
# vector native type
# ----------------------

# Recap sqlite3 DB connection
# import oracledb
#       ----------
# oracledb.connect(...) ->connection_object
# connection_object.cursor()
#..

# from langchain_oracledb.vectorstores import OracleVS

# FAISS.from_documents()
# -----
# OracleVS.from_documents(documents=<docs>,embeddings=<embObj>,client=<oracleDBConnection>,table_name="<tableName>")
# -------------------------- > vector_store
#


## The pipeline steps
# ------------------------
# 1. Connct to Oracle23ai use oracledb module
# 2. Generate Embeddings - use embedding model
# 3. Store the documents chunks and vectors in Oracle
# 4. Retrieve relevant chunks for a user question
# 5. Rerank ->LLM

In [104]:
vector_stores.similarity_search('self-attention',k=3)

[Document(id='baeda9fe-691e-4b9f-959c-6839e38c4420', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 4, 'page_label': '5'}, page_content='[38, 2, 9].\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values\nand queries come from the same place, in this case, the output of the previous layer in the\nencoder. Each position in the encoder can attend to all positions in the previous layer of the\nencoder.\n• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to\nall positions in the decoder up to and including that position. We need to prevent leftward'),
 Docum

In [105]:
vector_stores.similarity_search_with_score('self-attention',k=3)

[(Document(id='baeda9fe-691e-4b9f-959c-6839e38c4420', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-08-03T00:07:29+00:00', 'author': '', 'keywords': '', 'moddate': '2023-08-03T00:07:29+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'attention.pdf', 'total_pages': 15, 'page': 4, 'page_label': '5'}, page_content='[38, 2, 9].\n• The encoder contains self-attention layers. In a self-attention layer all of the keys, values\nand queries come from the same place, in this case, the output of the previous layer in the\nencoder. Each position in the encoder can attend to all positions in the previous layer of the\nencoder.\n• Similarly, self-attention layers in the decoder allow each position in the decoder to attend to\nall positions in the decoder up to and including that position. We need to prevent leftward'),
  np.

In [ ]:
################################  End of the DAY4 session ###############